# NAS - Top-K Beam Selection

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

### The Data We Have

**Data Summary**

- **beam_output**  
  - **Shape:** (9638, 8, 32)  
  - **Represents:** Target labels representing beam scores for each sample.  
  - **Example:** For one sample, a portion of the beam scores might look like:  
    ```
    [[1.51e-06, 1.76e-06, 3.01e-06, ...], 
     [1.77e-06, 2.02e-06, 3.37e-06, ...], 
     ...]
    ```  
    Each value is a float (with an imaginary part of zero), indicating the quality of a specific beam pair.

- **coord_input**  
  - **Shape:** (9638, 2)  
  - **Represents:** 2D coordinates associated with each sample (e.g., spatial positions).  
  - **Example:** The first sample might have coordinates similar to:  
    ```
    [748.92, 624.72]
    ```

- **image_input**  
  - **Shape:** (9638, 48, 81, 1)  
  - **Represents:** Grayscale image data where each pixel is an 8-bit unsigned integer.  
  - **Example:** A snippet from the first image might include pixel values such as:  
    ```
    [[[156], [136], [119], ...],
     [[131], [105], [91], ...],
     [[153], [116], [100], ...],
     ...]
    ```

- **lidar_input**  
  - **Shape:** (9638, 20, 200, 10)  
  - **Represents:** LIDAR data formatted as a multi-channel grid (20×200 with 10 channels), encoding spatial features or intensities. This representation indicates that LIDAR data contains spatial information about obstructions, base stations, and the target vehicle, which can be crucial for predicting beamforming paths.
  - **Semantic Meaning of Voxel Values:**  
    - **-2:** Base Station (BS) location  
    - **-1:** Target vehicle (receiver)  
    - **1:** Obstacles (e.g., other vehicles, buildings, pedestrians, trees)  
    - **0:** Empty space (free path for mmWave signals)  
  - **Example:**  
    ```
    [[[0, 0, 0, ..., 0, 0, 0],
      [0, 0, 0, ..., 0, 0, 0],
      ...,
      [-2, -1, 1, ..., 0, 0, 0]],
     ...]
    ```

## 1. Imports

In [3]:
# --------------------------- Standard Libraries ---------------------------- #
import os
import time
import psutil
import subprocess
import traceback
from threading import Thread

# -------------------------------- Annotations ------------------------------- #
from typing import List, Optional, Tuple

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np
import matplotlib.pyplot as plt

# ----------------------- TensorFlow and Keras Modules ---------------------- #
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# --------------------------------- AutoKeras -------------------------------- #
import autokeras as ak

2025-02-25 09:35:52.515496: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-25 09:35:52.527579: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740486952.540508   41597 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740486952.544450   41597 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-25 09:35:52.559251: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## 2. Utility Functions Definitions

### 2.1. GPU

In [4]:
def get_gpu_info():
    """
    Retrieves and prints detailed GPU information including TensorFlow,
    CUDA, cuDNN versions, number of GPUs, and memory details.
    """
    # Display TensorFlow version
    print(f"TensorFlow Version: {tf.__version__}")

    # Check if TensorFlow is built with CUDA support and retrieve build info
    if tf.test.is_built_with_cuda():
        build_info = tf.sysconfig.get_build_info()
        print(f"TensorFlow is built with CUDA support")
        print(f"CUDA Version: {build_info['cuda_version']}")
        print(f"cuDNN Version: {build_info['cudnn_version']}")
    else:
        print("Running on CPU (No CUDA support detected)")

    # Detect available GPUs
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"\nNumber of GPUs detected: {len(gpus)}")
        print(f"Available GPU(s): {[gpu.name for gpu in gpus]}\n")
        tf.test.gpu_device_name()
    else:
        print("No GPUs found")
        print("Running on CPU")

### 2.2. Folder and Files

In [5]:
def create_run_directory(prefix: str, base_dir: str = "runs") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        prefix (str): Prefix for the run directory.
        base_dir (str): Base directory for storing training runs. Defaults to "runs".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir


def save_trial_params_to_file(filepath: str, params: dict, **kwargs) -> None:
    """
    Saves the parameters of a trial along with additional information to a file.

    Args:
        filepath (str): Path to the file where parameters will be saved.
        params (dict): Parameters of the trial to save.
        **kwargs: Additional information to include in the file (e.g., rank, trial ID, loss).
    """
    with open(filepath, "w") as txt_file:
        # Write additional info
        for key, value in kwargs.items():
            txt_file.write(f"{key}: {value}\n")
        
        # Write parameters
        txt_file.write("Parameters:\n")
        for param_name, param_value in params.items():
            txt_file.write(f"  {param_name}: {param_value}\n")

### 2.3. Logging

In [6]:
def log_resources(log_dir: str, interval: int = 5, **kwargs) -> None:
    """
    Logs selected resources (CPU, RAM, GPU, CUDA, TensorFlow) at regular intervals.

    Args:
        log_dir (str): Directory to save log files.
        interval (int): Time interval between logs (seconds). Default is 5.
        kwargs: Resource options as boolean. Supported options: 
                "cpu", "ram", "gpu", "cuda", "tensorflow".
    """
    os.makedirs(log_dir, exist_ok=True)

    def log_cpu():
        log_path = os.path.join(log_dir, "cpu_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,CPU_Usage(%),Per-Core_Usage(%)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    cpu_usage = psutil.cpu_percent()
                    per_core_usage = psutil.cpu_percent(percpu=True)
                    f.write(f"{timestamp},{cpu_usage},{','.join(map(str, per_core_usage))}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_ram():
        log_path = os.path.join(log_dir, "ram_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,Total(MB),Used(MB),Free(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    mem = psutil.virtual_memory()
                    total = mem.total / (1024 ** 2)
                    used = mem.used / (1024 ** 2)
                    free = mem.available / (1024 ** 2)
                    f.write(f"{timestamp},{total:.2f},{used:.2f},{free:.2f}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_gpu():
        log_path = os.path.join(log_dir, "gpu_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,GPU_ID,Memory_Used(MB),Memory_Total(MB),GPU_Utilization(%)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    gpu_stats = subprocess.run(
                        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu",
                         "--format=csv,noheader,nounits"],
                        capture_output=True,
                        text=True,
                    ).stdout.strip()
                    
                    for line in gpu_stats.split("\n"):
                        gpu_id, mem_used, mem_total, util = map(int, line.split(","))
                        f.write(f"{timestamp},{gpu_id},{mem_used},{mem_total},{util}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_cuda():
        log_path = os.path.join(log_dir, "cuda_usage_log.csv")
        with open(log_path, "w") as f:
            f.write("Timestamp,Process_Memory_Used(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    cuda_mem_stats = subprocess.run(
                        ["nvidia-smi", "--query-compute-apps=used_memory", "--format=csv,noheader,nounits"],
                        capture_output=True,
                        text=True,
                    ).stdout.strip()
                    if cuda_mem_stats:
                        f.write(f"{timestamp},{cuda_mem_stats} MB\n")
                    else:
                        f.write(f"{timestamp},0 MB\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    def log_tensorflow():
        log_path = os.path.join(log_dir, "tensorflow_usage_log.csv")
        os.makedirs(log_dir, exist_ok=True)
        with open(log_path, "w") as f:
            f.write("Timestamp,Device,Memory_Allocated(MB),Memory_Peak(MB)\n")
            while True:
                try:
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                    gpus = tf.config.experimental.list_physical_devices("GPU")
                    for gpu in gpus:
                        device_name = gpu.name  # The correct device name
                        memory_info = tf.config.experimental.get_memory_info("GPU:0")
                        allocated_memory = memory_info["current"] / (1024 ** 2)  # Convert to MB
                        peak_memory = memory_info["peak"] / (1024 ** 2)  # Convert to MB
                        f.write(f"{timestamp},{device_name},{allocated_memory:.2f},{peak_memory:.2f}\n")
                    f.flush()
                    time.sleep(interval)
                except Exception as e:
                    f.write(f"Error: {e}\n")
                    break

    # Start logging threads based on user selection
    if kwargs.get("cpu", False):
        Thread(target=log_cpu, daemon=True).start()
    if kwargs.get("ram", False):
        Thread(target=log_ram, daemon=True).start()
    if kwargs.get("gpu", False):
        Thread(target=log_gpu, daemon=True).start()
    if kwargs.get("cuda", False):
        Thread(target=log_cuda, daemon=True).start()
    if kwargs.get("tensorflow", False):
        Thread(target=log_tensorflow, daemon=True).start()

## 3. Setup and Configuration

### 3.1. GPU Management

In [7]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

TensorFlow Version: 2.18.0
TensorFlow is built with CUDA support
CUDA Version: 12.5.1
cuDNN Version: 9

Number of GPUs detected: 1
Available GPU(s): ['/physical_device:GPU:0']



I0000 00:00:1740486954.507967   41597 gpu_device.cc:2022] Created device /device:GPU:0 with 2229 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


### 3.2. Folder and File Management

In [8]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = None  # "runs/nas_ak_2" 

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_ak_")

LOG_DIR = os.path.join(RUN_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Run directory: {RUN_DIR}")

Run directory: runs/nas_ak_12


## 4. Constants and Hyperparameters

In [9]:
# ---------------------------- Training Parameters --------------------------- #
NUM_TRIALS = 200
EPOCHS = None 
BATCH_SIZE = 32

# ------------------------- Randomization Parameters ------------------------- #
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 5. Data Loading and Preprocessing

### 5.1. Data Loading

In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
y_train = np.load(beam_output_path)
coord_input = np.load(coord_input_path)
image_input = np.load(image_input_path)
lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
y_train = y_train.astype(np.float32)
image_input = np.array(image_input, dtype=np.float32)
coord_input = np.array(coord_input, dtype=np.float32)
lidar_input = np.array(lidar_input, dtype=np.float32)

# Flatten y
y_train = y_train.reshape(y_train.shape[0], -1)

# Print the shapes of the loaded data
print(f"y_train shape: {y_train.shape}")
print(f"coord_input shape: {coord_input.shape}")
print(f"image_input shape: {image_input.shape}")
print(f"lidar_input shape: {lidar_input.shape}")

/tmp/ipykernel_41597/1013196349.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  y_train = y_train.astype(np.float32)


y_train shape: (9638, 256)
coord_input shape: (9638, 2)
image_input shape: (9638, 48, 81, 1)
lidar_input shape: (9638, 20, 200, 10)
y_train type: <class 'numpy.ndarray'>
coord_input type: <class 'numpy.ndarray'>
image_input type: <class 'numpy.ndarray'>
lidar_input type: <class 'numpy.ndarray'>


In [11]:
# ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of 100 samples for testing
# y_train = y_train[:SAMPLE_SIZE]
# coord_input = coord_input[:SAMPLE_SIZE]
# image_input = image_input[:SAMPLE_SIZE]
# lidar_input = lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train shape: {y_train.shape}")
# print(f"coord_input shape: {coord_input.shape}")
# print(f"image_input shape: {image_input.shape}")
# print(f"lidar_input shape: {lidar_input.shape}")

## 6. NAS Study

### 6.1. NAS Study Setup AK

In [12]:
def build_auto_model() -> ak.AutoModel:
    """Build and return an AutoKeras model that integrates multiple inputs for
    predicting beam scores.

    The model accepts three inputs:
      - image_input: Grayscale image with shape (48, 81, 1)
      - coord_input: 2D coordinates with shape (2,)
      - lidar_input: LIDAR data with shape (20, 200, 10)

    It outputs a regression prediction with shape (8, 32) corresponding to beam scores.

    Returns:
        ak.AutoModel: A compiled AutoKeras AutoModel configured for regression.
    """
    # Define each input node
    image_input = ak.ImageInput(shape=(48, 81, 1), name="image_input")
    coord_input = ak.Input(shape=(2,), name="coord_input")
    lidar_input = ak.Input(shape=(20, 200, 10), name="lidar_input")
    
    # Define the output wanted for the model
    output = ak.RegressionHead(output_dim=8*32, metrics=["mae"])
    
    automodel = ak.AutoModel(
        inputs=[image_input, coord_input, lidar_input],
        outputs=output,
        max_trials=NUM_TRIALS,
        overwrite=False, # Setting overwrite=False enables resume training
        directory=os.path.join(RUN_DIR, "auto_model"),
    )
    return automodel

### 6.2 Code Health Info

In [13]:
log_resources(
    log_dir=LOG_DIR,
    interval=5,
    cpu=False,
    ram=True,
    gpu=True,
    cuda=False,
    tensorflow=False,
)

In [14]:
# ---------------------------- Kernel life monitor --------------------------- #
#! Remove this block if you don't want to use it
try:
    pid_path = os.path.join(RUN_DIR, "kernel_pid.txt")
    pid = os.getpid()
    with open(pid_path, "w") as f:
        f.write(str(pid))
    print(f"[INFO] Kernel PID saved to {pid_path}: {pid}")
    print(f"Call the monitor script: python _monitor_kernel_life.py --pid_path {pid_path} --interval 10\n")

    # from _monitor_kernel_life import start_monitoring
    # start_monitoring(pid_path=pid_path, check_interval=5)
except Exception as e:
    print("[ERROR] Kernel monitoring failed!")
    print(e)
    pass

[INFO] Kernel PID saved to runs/nas_ak_12/kernel_pid.txt: 41597
Call the monitor script: python _monitor_kernel_life.py --pid_path runs/nas_ak_12/kernel_pid.txt --interval 10



### 6.3. Do the NAS

In [15]:
model_path = os.path.join(RUN_DIR, "models")
os.makedirs(model_path, exist_ok=True)

# -------------------------- Build and Train Model -------------------------- #
automodel = build_auto_model()
# Train the model; adjust epochs as needed based on convergence
automodel.fit(
    x=[image_input, coord_input, lidar_input],
    y=y_train, # Already flattened
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=None,
    validation_split=0.20,
    verbose=2,
)

# Save the trained model
best_model = automodel.export_model()
best_model.save(os.path.join(model_path, "best_model.keras"))

best_model.summary()

# ----------------------- Email api to notify the user ----------------------- #
#! Remove this block if you don't want to send an email
try:
    from utils.email_api import send_email

    # Message that the NAS Beam Selection study is complete
    email_subject = "NAS Beam Selection Study Complete"
    email_body = """
    <html>
        <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333; background-color: #f9f9f9; padding: 20px;">
        <div style="max-width: 600px; margin: auto; background: #fff; padding: 20px; border: 1px solid #ddd; border-radius: 8px;">
            <h2 style="color: #0056b3; text-align: center;">🎉 NAS Beam Selection Study Complete</h2>
            <p style="font-size: 16px; color: #444;">
            <strong>Dear User,</strong>
            </p>
            <p style="font-size: 18px; color: #333;">
            Your NAS Beam Selection study has successfully completed!
            </p>
            <p style="text-align: center; font-size: 16px;">
            <strong style="color: #28a745;">✔️ Study Status:</strong> <span style="color: #0056b3;">Completed</span>
            </p>
            <footer style="margin-top: 20px; text-align: center; font-size: 14px; color: #888;">
            <p>Best regards,</p>
            <p><strong>The Optimization Team</strong></p>
            </footer>
        </div>
        </body>
    </html>
    """
    send_email(
        subject=email_subject,
        body=email_body,
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        text_type="html",
    )
except Exception as e:
    print(f"[ERROR] Failed to send email: {e}")
    traceback.print_exc()
    pass

I0000 00:00:1740486955.430986   41597 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2229 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


KeyboardInterrupt: 